# Multi-Agent Orchestration for RAG Systems
# Step 1. RAG Baseline Reproduction

*Asignee: Alla* - *Review:*

This notebook reproduces baseline retrieval results for https://github.com/Trista1208/advanced_genAI.git (23.12.2025)
- BM25
- Dense Retrieval
- GraphRAG
- Hybrid Retrieval
- Re-ranking

It reports consistent metrics for all methods:
- Precision@k
- Recall@k
- MRR

Set `EVAL_SCOPE = 'full_corpus'` in the paths cell to run full-corpus evaluation.


In [16]:
# Colab setup
%pip install -q langchain-core langchain-community langchain-huggingface chromadb \
  rank-bm25 langdetect nltk sentence-transformers pytrec_eval


In [17]:
import os
import re
import json
import time
import pickle
import random
import pathlib
import importlib.util
from dataclasses import dataclass
from collections import defaultdict
from typing import Any

import numpy as np
import pandas as pd
import nltk
from langdetect import detect
from rank_bm25 import BM25Okapi
from tqdm.auto import tqdm

from sentence_transformers import SentenceTransformer
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)

STOP_EN = set(nltk.corpus.stopwords.words('english'))
STOP_DE = set(nltk.corpus.stopwords.words('german'))

print('Setup complete.')


Setup complete.


In [18]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1.1 Setup

You need a hugging face token, https://huggingface.co/settings/tokens, add `HF_TOKEN` to Colab Secrets

To have all the necessary data in the right place you need to create a folder Adv_GenAI in your Google Drive and copy folders `benchmark` and `storage` **(and its content).** The respective paths are `/content/drive/MyDrive/Adv_GenAI/benchmark`
and `/content/drive/MyDrive/Adv_GenAI/storage` Copying took me around 20 minutes.


Following code can be run using full corpus or a sample of it. We report evaluation scores of both options, similar how the group from previous semester was doing it.
You can change it by changing a variable:

`EVAL_SCOPE = 'full_corpus' or 'subsample'`




In [19]:
# Paths (edit PROJECT_ROOT if needed)
# PROJECT_ROOT must be the folder that contains benchmark/ and storage/
CANDIDATE_ROOTS = [
    pathlib.Path('/content/drive/MyDrive/Adv_GenAI'),
    pathlib.Path('/content/drive/MyDrive/advanced-genai-26/baseline/advanced_genAI-main/data'),
    pathlib.Path('/content/drive/MyDrive/advanced_genAI-main/data'),
]

def looks_like_project_root(p: pathlib.Path) -> bool:
    return (p / 'benchmark').exists() and (p / 'storage').exists()

PROJECT_ROOT = None
for c in CANDIDATE_ROOTS:
    if looks_like_project_root(c):
        PROJECT_ROOT = c
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not auto-detect project root. Set PROJECT_ROOT manually.')

PROJECT_ROOT = PROJECT_ROOT.resolve()
print('PROJECT_ROOT =', PROJECT_ROOT)

# Choose scope: 'subsample' or 'full_corpus'
EVAL_SCOPE = 'full_corpus'
assert EVAL_SCOPE in {'subsample', 'full_corpus'}
print('EVAL_SCOPE =', EVAL_SCOPE)

PATH_QA = PROJECT_ROOT / 'benchmark/benchmark_qa_bilingual.json'
PATH_QRELS_FIXED = PROJECT_ROOT / 'benchmark/score/fixed_size'

if EVAL_SCOPE == 'subsample':
    PATH_BM25_PICKLE = PROJECT_ROOT / 'storage/subsample/retrieval_downstream/bm25_fixed_qe.pkl'
    if not PATH_BM25_PICKLE.exists():
        PATH_BM25_PICKLE = PROJECT_ROOT / 'storage/subsample/retrieval/fixed_size_chunk/bm25_retriever.pkl'
    PATH_DENSE_INDEX = PROJECT_ROOT / 'storage/subsample/vectordb_dense/fixed_e5'
    PATH_GRAG_ROOT = PROJECT_ROOT / 'storage/subsample/retrieval_graph'
    PATH_CHUNK_PKL = PROJECT_ROOT / 'storage/subsample/Lang_norm/fixed_size_chunk/docs_fixed_norm.pkl'
else:
    PATH_BM25_PICKLE = PROJECT_ROOT / 'storage/full_corpus/retrieval/fixed_size_chunk/bm25_retriever_full.pkl'
    PATH_DENSE_INDEX = PROJECT_ROOT / 'storage/full_corpus/vectordb_dense/fixed_e5'
    PATH_GRAG_ROOT = PROJECT_ROOT / 'storage/full_corpus/retrieval_graph'
    PATH_CHUNK_PKL = PROJECT_ROOT / 'storage/full_corpus/Lang_norm/fixed_size_chunk/docs_fixed_norm.pkl'

OUT_DIR = PROJECT_ROOT / f'results/baseline_repro_colab_{EVAL_SCOPE}'
OUT_DIR.mkdir(parents=True, exist_ok=True)

required = [
    PATH_BM25_PICKLE, PATH_QA, PATH_QRELS_FIXED,
    PATH_DENSE_INDEX, PATH_GRAG_ROOT, PATH_CHUNK_PKL
]
for p in required:
    if not p.exists():
        raise FileNotFoundError(f'Missing required path: {p}')

print('All required artifacts found.')


PROJECT_ROOT = /content/drive/MyDrive/Adv_GenAI
EVAL_SCOPE = full_corpus
All required artifacts found.


## 1.2 Evaluation Tools: BM25
**We start with preparing our own tools.** By adding small wrappers, we ensure that each method can be called in the same way and evaluated with the same metric pipeline, which keeps the baseline comparison consistent and technically fair.

Next cell prepares a compatibility **layer for BM25** so that pickled retrievers from different stages of the project can be used in one reproducible evaluation pipeline. In our setup, subsample and full-corpus artifacts are not always serialized with the same internal structure, so we load the original object and wrap it with a unified adapter that exposes a consistent search(query, top_k) interface. This avoids format-specific failures and ensures that BM25 can be evaluated with exactly the same downstream metric code as Dense, GraphRAG, Hybrid, and Re-ranking methods.

In [20]:
# Robust BM25 loader for both subsample and full-corpus pickle formats
class BilingualBM25:
    """Compatibility class for notebook pickles."""

    def _rank_lang(self, q: str, lang: str, k: int):
        # Subsample-style object: self.bm25 + self.docs_by_lang
        try:
            q_tokens = nltk.word_tokenize(q)
        except Exception:
            q_tokens = q.split()
        scores = self.bm25[lang].get_scores(q_tokens)
        idx = np.argsort(scores)[::-1][:k]
        hits = []
        for i in idx:
            d = self.docs_by_lang[lang][i]
            d.metadata['bm25_score'] = float(scores[i])
            hits.append(d)
        return hits

    def _get_docs_with_scores(self, ret, qq, top_k):
        # Full-corpus-style object: self.retrievers
        if hasattr(ret, 'get_relevant_documents_with_scores'):
            try:
                return ret.get_relevant_documents_with_scores(qq, k=top_k)
            except Exception:
                pass

        if hasattr(ret, 'vectorizer') and hasattr(ret, 'docs'):
            try:
                toks = qq.lower().split()
                scores = ret.vectorizer.get_scores(toks)
                ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)[:top_k]
                return [(ret.docs[idx], float(score)) for idx, score in ranked]
            except Exception:
                pass

        if hasattr(ret, 'invoke'):
            try:
                old_k = getattr(ret, 'k', None)
                if old_k is not None:
                    ret.k = top_k
                docs = ret.invoke(qq)
                if old_k is not None:
                    ret.k = old_k
                return [(d, d.metadata.get('score', 0.0)) for d in docs[:top_k]]
            except Exception:
                pass

        return []

    def search(self, query: str, top_k: int = 100):
        # Route to correct behavior based on available attributes
        if hasattr(self, 'bm25') and hasattr(self, 'docs_by_lang'):
            src = detect(query) if query.strip() else 'en'
            src = src if src in ('en', 'de') else 'en'
            bag = []
            translator = getattr(self, 'translator', None)
            for lang in ('en', 'de'):
                q_lang = translator.translate(query, lang) if translator and lang != src else query
                bag.extend(self._rank_lang(q_lang, lang, top_k))

            best = {}
            for d in bag:
                uid = d.metadata.get('chunk_id') or d.metadata.get('record_id')
                if uid not in best or d.metadata['bm25_score'] > best[uid].metadata.get('bm25_score', -1e9):
                    best[uid] = d
            return sorted(best.values(), key=lambda d: d.metadata.get('bm25_score', 0.0), reverse=True)[:top_k]

        if hasattr(self, 'retrievers') and isinstance(self.retrievers, dict):
            src = detect(query) if query.strip() else 'en'
            src = src if src in ('en', 'de') else 'en'
            bag = []
            translator = getattr(self, 'translator', None)

            for lang, ret in self.retrievers.items():
                qq = translator.translate(query, lang) if translator and lang != src else query
                docs_with_scores = self._get_docs_with_scores(ret, qq, top_k)
                for doc, score in docs_with_scores:
                    doc.metadata['bm25_score'] = float(score)
                    bag.append(doc)

            best = {}
            for d in bag:
                uid = d.metadata.get('chunk_id') or d.metadata.get('record_id')
                if uid is None:
                    continue
                if uid not in best or d.metadata.get('bm25_score', -1e9) > best[uid].metadata.get('bm25_score', -1e9):
                    best[uid] = d

            return sorted(best.values(), key=lambda d: d.metadata.get('bm25_score', 0.0), reverse=True)[:top_k]

        raise AttributeError('Unsupported BilingualBM25 object format.')

class QEBM25:
    @staticmethod
    def _expand_query(query: str, base_retriever, fb_docs: int = 5, fb_terms: int = 5) -> str:
        def tok(text: str):
            try:
                return nltk.word_tokenize(text.lower())
            except Exception:
                return text.lower().split()

        hits = base_retriever.search(query, top_k=fb_docs)
        tokens = [
            t for h in hits for t in tok(h.page_content)
            if t.isalpha() and t not in STOP_EN and t not in STOP_DE
        ]
        extra = ' '.join(w for w, _ in nltk.FreqDist(tokens).most_common(fb_terms))
        return f'{query} {extra}' if extra else query

    def search(self, query: str, top_k: int = 100):
        if hasattr(self, 'base'):
            expanded = self._expand_query(query, self.base)
            return self.base.search(expanded, top_k)
        raise AttributeError('QEBM25 object missing base retriever.')

with open(PATH_BM25_PICKLE, 'rb') as f:
    bm25_raw = pickle.load(f)

class BM25RetrieverAdapter:
    def __init__(self, obj):
        self.obj = obj

    def search(self, query: str, top_k: int = 100):
        # Primary path
        if hasattr(self.obj, 'search'):
            try:
                return self.obj.search(query, top_k=top_k)
            except TypeError:
                return self.obj.search(query, k=top_k)

        # LangChain retriever fallback
        if hasattr(self.obj, 'invoke'):
            old_k = getattr(self.obj, 'k', None)
            if old_k is not None:
                self.obj.k = top_k
            docs = self.obj.invoke(query)
            if old_k is not None:
                self.obj.k = old_k
            for rank, d in enumerate(docs, start=1):
                if hasattr(d, 'metadata'):
                    d.metadata.setdefault('bm25_score', float(top_k - rank))
            return docs[:top_k]

        raise AttributeError(f'Unsupported BM25 object type: {type(self.obj)}')

bm25_retriever = BM25RetrieverAdapter(bm25_raw)
print('BM25 loaded:', type(bm25_raw), '-> adapter ready')


BM25 loaded: <class '__main__.BilingualBM25'> -> adapter ready


## 1.3 Evaluation Tools: Dense Retriever
The next cell initializes the **Dense retriever** by loading the multilingual E5 embedding model and attaching it to the persisted Chroma index for the selected evaluation scope. Queries are formatted with the query: prefix expected by E5, and retrieved vector distances are converted into similarity scores for consistent ranking behavior. This gives us a semantic retrieval baseline that is directly comparable to BM25 in the shared evaluation framework.

In [21]:
# Dense retriever
class DenseRetriever:
    def __init__(self, index_dir: pathlib.Path, model_name='intfloat/multilingual-e5-large-instruct', k: int = 100):
        self.k = k
        self.embeddings = HuggingFaceEmbeddings(
            model_name=model_name,
            model_kwargs={'device': 'cuda' if os.path.exists('/proc/driver/nvidia/version') else 'cpu'},
            encode_kwargs={'batch_size': 32, 'normalize_embeddings': True},
        )
        self.store = Chroma(persist_directory=str(index_dir), embedding_function=self.embeddings)

    def _prep(self, q: str) -> str:
        return 'query: ' + q.strip()

    def search(self, query: str, top_k: int = 100):
        k = top_k or self.k
        hits = self.store.similarity_search_with_score(self._prep(query), k=k)
        out = []
        for doc, dist in hits:
            doc.metadata['dense_score'] = 1.0 - float(dist)
            out.append(doc)
        return out

dense_retriever = DenseRetriever(PATH_DENSE_INDEX, k=100)
print('Dense retriever ready.')


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Dense retriever ready.


## 1.4 Evaluation Tools: GraphRAG retriever
This cell defines and initializes the **GraphRAG retriever** by loading precomputed community embeddings, the community-to-chunk mapping, and the chunk corpus. At query time, it first selects the most relevant graph communities, then scores candidate chunks within those communities using embedding similarity, and returns the top-ranked documents with normalized GraphRAG scores. This provides a graph-informed retrieval baseline that complements lexical and dense retrieval in the final comparison.

In [22]:
# GraphRAG retriever
class GraphRAGRetriever:
    def __init__(self, graph_root: pathlib.Path, chunk_pkl: pathlib.Path):
        self.root = graph_root
        self.emb_dir = graph_root / 'embeddings'
        self.chunk_pkl = chunk_pkl
        self.embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
        self._emb_cache = {}
        self._cid_cache = {}
        self._chunk_by_id = None
        self._chunk_vec_cache = {}
        self.comm2chunk = json.loads((self.root / 'comm2chunk_fixed.json').read_text(encoding='utf-8'))

    def _load_embeddings(self, level: int):
        if level in self._emb_cache:
            return self._emb_cache[level], self._cid_cache[level]
        mat = np.load(self.emb_dir / f'EMB_fixed_C{level}.npy')
        cid = json.loads((self.emb_dir / f'CID_fixed_C{level}.json').read_text(encoding='utf-8'))
        self._emb_cache[level] = mat
        self._cid_cache[level] = cid
        return mat, cid

    def _load_chunks(self):
        if self._chunk_by_id is not None:
            return self._chunk_by_id
        with open(self.chunk_pkl, 'rb') as f:
            docs_norm = pickle.load(f)

        def restore(d):
            raw = d.metadata.get('original_text') or d.page_content
            return Document(page_content=raw, metadata=d.metadata)

        docs = [restore(d) for d in docs_norm]
        self._chunk_by_id = {d.metadata['chunk_id']: d for d in docs}
        return self._chunk_by_id

    def _chunk_vec(self, cid: str, chunks: dict):
        if cid not in self._chunk_vec_cache:
            self._chunk_vec_cache[cid] = self.embedder.encode([chunks[cid].page_content], normalize_embeddings=True)[0]
        return self._chunk_vec_cache[cid]

    def retrieve(self, query: str, level: str = 'C1', k_comms: int = 24, top_k: int = 100):
        L = int(level.lstrip('C'))
        emb_mat, cid_list = self._load_embeddings(L)
        chunks = self._load_chunks()

        q_vec = self.embedder.encode([query], normalize_embeddings=True)[0]
        sims_comm = emb_mat @ q_vec
        best_idx = sims_comm.argsort()[::-1][:k_comms]

        cand_ids = set()
        for idx in best_idx:
            cand_ids.update(self.comm2chunk.get(cid_list[idx], []))

        scored = []
        for cid in cand_ids:
            if cid not in chunks:
                continue
            sim = float(self._chunk_vec(cid, chunks) @ q_vec)
            scored.append((cid, sim))

        scored.sort(key=lambda x: x[1], reverse=True)
        scored = scored[:top_k]

        out = []
        for cid, sim in scored:
            d = chunks[cid]
            d.metadata['grag_score'] = (sim + 1.0) / 2.0
            out.append(d)
        return out

    def search(self, query: str, top_k: int = 100):
        return self.retrieve(query=query, level='C1', k_comms=24, top_k=top_k)

graph_retriever = GraphRAGRetriever(PATH_GRAG_ROOT, PATH_CHUNK_PKL)
print('GraphRAG retriever ready.')


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GraphRAG retriever ready.


##1.5 Evaluation Tools: Hybrid Retrieval and Reranking
This cell defines the combined retrieval baselines used after the individual retrievers are loaded. Hybrid retrieval fuses BM25, Dense, and GraphRAG candidate lists with weighted reciprocal rank fusion to produce a single ranked output, while the re-ranking variant applies an additional overlap-based reranking step on top of fused candidates to improve top-ranked relevance. This allows us to evaluate both plain fusion and post-fusion refinement within the same framework.

In [23]:
# Hybrid + Re-ranking
def _uid(doc: Any):
    meta = getattr(doc, 'metadata', {}) or {}
    return meta.get('chunk_id') or meta.get('record_id') or meta.get('doc_id')

def _safe_unique(docs):
    out, seen = [], set()
    for d in docs:
        u = _uid(d)
        if u is None or u in seen:
            continue
        seen.add(u)
        out.append(d)
    return out

def _rrf_fuse(runs: dict, k_rrf: int = 60, weights=None):
    weights = weights or {'bm25': 1.2, 'dense': 1.0, 'graph': 0.6}
    scores = defaultdict(float)
    store = {}
    for name, docs in runs.items():
        w = float(weights.get(name, 1.0))
        for rank, d in enumerate(docs, start=1):
            u = _uid(d)
            if u is None:
                continue
            store.setdefault(u, d)
            scores[u] += w * (1.0 / (k_rrf + rank))
    fused = sorted(store.values(), key=lambda d: scores[_uid(d)], reverse=True)
    for d in fused:
        d.metadata['fused_score'] = float(scores[_uid(d)])
    return fused

def _overlap_rerank(docs, query: str, top_k: int):
    q_terms = set(t.lower() for t in query.split() if t.strip())
    scored = []
    for d in docs:
        text = (d.metadata.get('original_text') or d.page_content or '').lower()
        overlap = len(q_terms & set(text.split())) / max(len(q_terms), 1)
        scored.append((overlap, d))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [d for _, d in scored[:top_k]]

class HybridRetriever:
    def __init__(self, bm25, dense, graph, rerank=False):
        self.bm25 = bm25
        self.dense = dense
        self.graph = graph
        self.rerank = rerank

    def search(self, query: str, top_k: int = 100):
        pre_k = max(30, top_k)
        bm = _safe_unique(self.bm25.search(query, top_k=pre_k))
        de = _safe_unique(self.dense.search(query, top_k=pre_k))
        gr = _safe_unique(self.graph.search(query, top_k=pre_k))

        fused = _rrf_fuse({'bm25': bm, 'dense': de, 'graph': gr})
        if self.rerank:
            fused = _overlap_rerank(fused[:max(50, top_k)], query, top_k=max(50, top_k))
        return fused[:top_k]

hybrid_retriever = HybridRetriever(bm25_retriever, dense_retriever, graph_retriever, rerank=False)
rerank_retriever = HybridRetriever(bm25_retriever, dense_retriever, graph_retriever, rerank=True)
print('Hybrid and ReRank retrievers ready.')


Hybrid and ReRank retrievers ready.


## 1.5 Evaluation
This cell runs the shared evaluation pipeline for all baseline methods using the same QA set and qrels. For each query, it collects ranked document IDs, computes Precision@k, Recall@k, and reciprocal rank, and then aggregates results into per-query and per-method summary tables. Using one evaluation function for all methods ensures the reported baseline comparison is consistent and directly comparable.

In [24]:
# Evaluation
K_VALUES = [1, 3, 5, 10]
TOP_N = 100

def load_qrels(folder: pathlib.Path, threshold: float = 0.5):
    qrels = defaultdict(set)
    for fp in sorted(folder.glob('*.json')):
        did = fp.stem
        payload = json.loads(fp.read_text(encoding='utf-8'))
        for qid, rel in payload.items():
            if float(rel.get('relevance_score', 0.0)) >= threshold:
                qrels[str(qid)].add(did)
    return qrels

def normalize_docs(docs, top_n=100):
    out, seen = [], set()
    for d in docs:
        did = _uid(d)
        if did is None or did in seen:
            continue
        out.append(str(did))
        seen.add(did)
        if len(out) >= top_n:
            break
    return out

def precision_at_k(ranked, relevant, k):
    top = ranked[:k]
    return sum(1 for x in top if x in relevant) / max(k, 1)

def recall_at_k(ranked, relevant, k):
    if not relevant:
        return 0.0
    top = ranked[:k]
    return sum(1 for x in top if x in relevant) / len(relevant)

def reciprocal_rank(ranked, relevant):
    for i, x in enumerate(ranked, start=1):
        if x in relevant:
            return 1.0 / i
    return 0.0

qa_data = json.loads(PATH_QA.read_text(encoding='utf-8'))
qrels = load_qrels(PATH_QRELS_FIXED)

methods = {
    'BM25': bm25_retriever,
    'Dense': dense_retriever,
    'GraphRAG': graph_retriever,
    'Hybrid': hybrid_retriever,
    'ReRank': rerank_retriever,
}

runs = {}
for name, retriever in methods.items():
    run = {}
    for q in tqdm(qa_data, desc=f'Running {name}'):
        qid = str(q['id'])
        docs = retriever.search(q['question'], top_k=TOP_N)
        run[qid] = normalize_docs(docs, top_n=TOP_N)
    runs[name] = run

per_query_rows = []
summary_rows = []

for name, run in runs.items():
    qids = sorted(set(run.keys()) & set(qrels.keys()))
    mrr_vals = []
    p_vals = {k: [] for k in K_VALUES}
    r_vals = {k: [] for k in K_VALUES}

    for qid in qids:
        ranked = run[qid]
        rel = qrels[qid]
        row = {'method': name, 'qid': qid}

        rr = reciprocal_rank(ranked, rel)
        row['MRR'] = rr
        mrr_vals.append(rr)

        for k in K_VALUES:
            pk = precision_at_k(ranked, rel, k)
            rk = recall_at_k(ranked, rel, k)
            row[f'Precision@{k}'] = pk
            row[f'Recall@{k}'] = rk
            p_vals[k].append(pk)
            r_vals[k].append(rk)

        per_query_rows.append(row)

    summary = {'method': name, 'queries_evaluated': len(qids), 'MRR': float(np.mean(mrr_vals) if mrr_vals else 0.0)}
    for k in K_VALUES:
        summary[f'Precision@{k}'] = float(np.mean(p_vals[k]) if p_vals[k] else 0.0)
        summary[f'Recall@{k}'] = float(np.mean(r_vals[k]) if r_vals[k] else 0.0)
    summary_rows.append(summary)

per_query_df = pd.DataFrame(per_query_rows)
summary_df = pd.DataFrame(summary_rows).sort_values('MRR', ascending=False).reset_index(drop=True)

summary_df


Running BM25:   0%|          | 0/25 [00:00<?, ?it/s]

Running Dense:   0%|          | 0/25 [00:00<?, ?it/s]

Running GraphRAG:   0%|          | 0/25 [00:00<?, ?it/s]

Running Hybrid:   0%|          | 0/25 [00:00<?, ?it/s]

Running ReRank:   0%|          | 0/25 [00:00<?, ?it/s]

,method,queries_evaluated,MRR,Precision@1,Recall@1,Precision@3,Recall@3,Precision@5,Recall@5,Precision@10,Recall@10
0,GraphRAG,24,0.232573,0.083333,0.000687,0.097222,0.003166,0.116667,0.005892,0.116667,0.052857
1,ReRank,24,0.222952,0.041667,0.000196,0.138889,0.004480,0.125000,0.006323,0.100000,0.010619
2,Hybrid,24,0.202154,0.000000,0.000000,0.097222,0.003699,0.125000,0.028352,0.095833,0.031149
3,Dense,24,0.165525,0.041667,0.000147,0.069444,0.008953,0.058333,0.010095,0.066667,0.034097
4,BM25,24,0.151296,0.041667,0.001016,0.055556,0.001681,0.091667,0.005506,0.091667,0.011165


On the full-corpus setup, GraphRAG achieved the best overall ranking quality with the highest MRR (0.233), indicating that graph-guided retrieval was most effective at placing relevant evidence early in the ranked list. ReRank and Hybrid followed closely, with ReRank slightly outperforming Hybrid on MRR, which suggests that post-fusion refinement can improve early precision in some cases. Dense and BM25 showed lower MRR, indicating weaker top-rank relevance when used alone in this setting. Overall, the results suggest that full-corpus retrieval benefits from multi-source or graph-aware methods, while single-retriever baselines remain useful reference points but are less competitive at early-rank retrieval quality.

In [25]:
# Save deliverables
summary_path = OUT_DIR / 'metrics_summary.csv'
per_query_path = OUT_DIR / 'metrics_per_query.csv'
runs_path = OUT_DIR / 'runs.json'

summary_df.to_csv(summary_path, index=False)
per_query_df.to_csv(per_query_path, index=False)
runs_path.write_text(json.dumps(runs, indent=2), encoding='utf-8')

print('Saved:')
print('-', summary_path)
print('-', per_query_path)
print('-', runs_path)


Saved:
- /content/drive/MyDrive/Adv_GenAI/results/baseline_repro_colab_full_corpus/metrics_summary.csv
- /content/drive/MyDrive/Adv_GenAI/results/baseline_repro_colab_full_corpus/metrics_per_query.csv
- /content/drive/MyDrive/Adv_GenAI/results/baseline_repro_colab_full_corpus/runs.json


## 1.6 Optional: Orchestration Evaluation (Separate from Baseline)

This section evaluates multi-retriever orchestration strategies separately from baseline methods:
- Voting
- Waterfall
- Confidence

These are reported in a separate table so baseline and orchestration analyses are not mixed.


In [27]:
# Orchestration retrievers
def _token_set(text: str):
    text = (text or '').lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    return set(t for t in text.split() if t)

def _top_overlap(query: str, docs):
    if not docs:
        return 0.0
    q_terms = _token_set(query)
    top_text = docs[0].metadata.get('original_text') or docs[0].page_content or ''
    d_terms = _token_set(top_text)
    return len(q_terms & d_terms) / max(len(q_terms), 1)

class VotingRetriever:
    name = 'Voting'
    def search(self, query, top_k=100):
        pre_k = max(30, top_k)
        bm = _safe_unique(bm25_retriever.search(query, top_k=pre_k))
        de = _safe_unique(dense_retriever.search(query, top_k=pre_k))
        gr = _safe_unique(graph_retriever.search(query, top_k=pre_k))
        fused = _rrf_fuse({'bm25': bm, 'dense': de, 'graph': gr}, weights={'bm25': 1.2, 'dense': 1.0, 'graph': 0.6})
        return fused[:top_k]

class WaterfallRetriever:
    name = 'Waterfall'
    def search(self, query, top_k=100):
        pre_k = max(30, top_k)
        bm = _safe_unique(bm25_retriever.search(query, top_k=pre_k))
        de = _safe_unique(dense_retriever.search(query, top_k=pre_k))
        fused = _rrf_fuse({'bm25': bm, 'dense': de, 'graph': []}, weights={'bm25': 1.2, 'dense': 1.0, 'graph': 0.0})
        docs = fused[:top_k]

        # Fallback: if lexical overlap is too low, include GraphRAG
        if _top_overlap(query, docs) < 0.05:
            gr = _safe_unique(graph_retriever.search(query, top_k=pre_k))
            fused2 = _rrf_fuse({'bm25': bm, 'dense': de, 'graph': gr}, weights={'bm25': 1.1, 'dense': 1.0, 'graph': 0.7})
            return fused2[:top_k]

        return docs

class ConfidenceRetriever:
    name = 'Confidence'

    @staticmethod
    def _route_weights(query: str):
        q = query.lower().strip()
        factoid = any(q.startswith(w) for w in ['who', 'when', 'where']) or any(ch.isdigit() for ch in q)
        semantic = any(w in q for w in ['why', 'how', 'explain', 'difference', 'impact'])

        if factoid:
            return {'bm25': 1.4, 'dense': 0.9, 'graph': 0.5}
        if semantic:
            return {'bm25': 0.9, 'dense': 1.3, 'graph': 0.6}
        return {'bm25': 1.0, 'dense': 1.1, 'graph': 0.5}

    def search(self, query, top_k=100):
        pre_k = max(30, top_k)
        bm = _safe_unique(bm25_retriever.search(query, top_k=pre_k))
        de = _safe_unique(dense_retriever.search(query, top_k=pre_k))
        gr = _safe_unique(graph_retriever.search(query, top_k=pre_k))
        weights = self._route_weights(query)
        fused = _rrf_fuse({'bm25': bm, 'dense': de, 'graph': gr}, weights=weights)
        return fused[:top_k]

orchestration_methods = {
    'Waterfall': WaterfallRetriever(),
    'Voting': VotingRetriever(),
    'Confidence': ConfidenceRetriever(),
}

print('Orchestration methods ready:', list(orchestration_methods.keys()))


Orchestration methods ready: ['Waterfall', 'Voting', 'Confidence']


In [28]:
# Evaluate orchestration methods with the same shared evaluator
orch_runs = {}
for name, retriever in orchestration_methods.items():
    run = {}
    for q in tqdm(qa_data, desc=f'Running {name}'):
        qid = str(q['id'])
        docs = retriever.search(q['question'], top_k=TOP_N)
        run[qid] = normalize_docs(docs, top_n=TOP_N)
    orch_runs[name] = run

orch_rows = []
for name, run in orch_runs.items():
    qids = sorted(set(run.keys()) & set(qrels.keys()))
    summary = {'method': name, 'queries_evaluated': len(qids)}

    mrr_vals = []
    p_vals = {k: [] for k in K_VALUES}
    r_vals = {k: [] for k in K_VALUES}

    for qid in qids:
        ranked = run[qid]
        rel = qrels[qid]
        mrr_vals.append(reciprocal_rank(ranked, rel))
        for k in K_VALUES:
            p_vals[k].append(precision_at_k(ranked, rel, k))
            r_vals[k].append(recall_at_k(ranked, rel, k))

    summary['MRR'] = float(np.mean(mrr_vals) if mrr_vals else 0.0)
    for k in K_VALUES:
        summary[f'Precision@{k}'] = float(np.mean(p_vals[k]) if p_vals[k] else 0.0)
        summary[f'Recall@{k}'] = float(np.mean(r_vals[k]) if r_vals[k] else 0.0)

    orch_rows.append(summary)

orch_summary_df = pd.DataFrame(orch_rows).sort_values('MRR', ascending=False).reset_index(drop=True)
orch_summary_df


Running Waterfall:   0%|          | 0/25 [00:00<?, ?it/s]

Running Voting:   0%|          | 0/25 [00:00<?, ?it/s]

Running Confidence:   0%|          | 0/25 [00:00<?, ?it/s]

,method,queries_evaluated,MRR,Precision@1,Recall@1,Precision@3,Recall@3,Precision@5,Recall@5,Precision@10,Recall@10
0,Confidence,24,0.208827,0.000000,0.000000,0.111111,0.003891,0.133333,0.028380,0.100000,0.031569
1,Waterfall,24,0.208303,0.041667,0.001344,0.138889,0.005348,0.100000,0.006052,0.070833,0.029732
2,Voting,24,0.202154,0.000000,0.000000,0.097222,0.003699,0.125000,0.028352,0.095833,0.031149


For orchestration on the full corpus, Confidence achieved the best MRR (0.209), with Waterfall very close (0.208) and Voting slightly lower (0.202), so overall early-rank performance is similar across all three strategies. Waterfall produced the strongest Precision@3 (0.139), indicating better short-list relevance in the top few results, while Confidence led at Precision@5 (0.133). At larger cutoffs, Confidence and Voting were nearly tied on Recall@10, with Waterfall slightly behind. In practice, these results suggest that orchestration variants are competitive but not dramatically separated, with Confidence showing the most balanced behavior and Waterfall favoring early precision at small k.

In [29]:
# Save separate orchestration results
orch_summary_path = OUT_DIR / 'metrics_orchestration_summary.csv'
orch_runs_path = OUT_DIR / 'runs_orchestration.json'

orch_summary_df.to_csv(orch_summary_path, index=False)
orch_runs_path.write_text(json.dumps(orch_runs, indent=2), encoding='utf-8')

print('Saved:')
print('-', orch_summary_path)
print('-', orch_runs_path)


Saved:
- /content/drive/MyDrive/Adv_GenAI/results/baseline_repro_colab_full_corpus/metrics_orchestration_summary.csv
- /content/drive/MyDrive/Adv_GenAI/results/baseline_repro_colab_full_corpus/runs_orchestration.json


## 1.7 Evaluation Full Corpus vs. Subsample
The **subsample setup contains 817 fixed-size chunks,** whereas the **full-corpus setup contains 7,531 fixed-size chunks.** This means the full corpus is roughly nine times larger, creating a substantially harder retrieval environment with many more candidate passages and potential distractors.

In [30]:
sub = pd.read_csv(PROJECT_ROOT / "results/baseline_repro_colab_subsample/metrics_summary.csv")
full = pd.read_csv(PROJECT_ROOT / "results/baseline_repro_colab_full_corpus/metrics_summary.csv")

sub["scope"] = "subsample"
full["scope"] = "full_corpus"

both = pd.concat([sub, full], ignore_index=True)

cols = ["scope", "method", "MRR", "Precision@1", "Precision@3", "Precision@5", "Precision@10",
        "Recall@1", "Recall@3", "Recall@5", "Recall@10"]
both = both[cols].sort_values(["method", "scope"])
both


,scope,method,MRR,Precision@1,Precision@3,Precision@5,Precision@10,Recall@1,Recall@3,Recall@5,Recall@10
9,full_corpus,BM25,0.151296,0.041667,0.055556,0.091667,0.091667,0.001016,0.001681,0.005506,0.011165
4,subsample,BM25,0.396272,0.166667,0.305556,0.316667,0.295833,0.002081,0.010785,0.017426,0.037034
8,full_corpus,Dense,0.165525,0.041667,0.069444,0.058333,0.066667,0.000147,0.008953,0.010095,0.034097
1,subsample,Dense,0.540476,0.375000,0.402778,0.350000,0.304167,0.031450,0.064326,0.073926,0.093394
5,full_corpus,GraphRAG,0.232573,0.083333,0.097222,0.116667,0.116667,0.000687,0.003166,0.005892,0.052857
0,subsample,GraphRAG,0.579613,0.458333,0.444444,0.366667,0.337500,0.029346,0.059344,0.063712,0.080849
7,full_corpus,Hybrid,0.202154,0.000000,0.097222,0.125000,0.095833,0.000000,0.003699,0.028352,0.031149
2,subsample,Hybrid,0.462108,0.250000,0.347222,0.316667,0.316667,0.004266,0.062629,0.076968,0.097059
6,full_corpus,ReRank,0.222952,0.041667,0.138889,0.125000,0.100000,0.000196,0.004480,0.006323,0.010619
3,subsample,ReRank,0.442001,0.291667,0.250000,0.233333,0.245833,0.002821,0.006986,0.012018,0.029840


Across all methods, performance is consistently higher on the subsample than on the full corpus, with large MRR drops when moving to full-corpus retrieval. The strongest relative degradation appears in Dense and GraphRAG, which perform very well on subsample but lose substantial early-rank quality at full scale, indicating that larger search space and higher distractor density make relevance ranking harder. BM25 also declines, but less dramatically in relative terms, remaining a stable lexical baseline with low absolute performance in both scopes. Hybrid and ReRank remain competitive in full-corpus settings and outperform single BM25/Dense in several top-k metrics, suggesting fusion and reranking help recover robustness under scale. Overall, the comparison confirms that subsample results are optimistic, while full-corpus evaluation is more realistic and should be treated as the primary indicator of deployment-level retrieval difficulty.

##1.8 Reproducibility Comparison Table


| Aspect | Original Report (PDF) | Reproduced Notebook (Current) | Match Status |
|---|---|---|---|
| Subsample size | 817 docs/chunks | 817 fixed-size chunks | Match |
| Full-corpus size | 7,544 docs/chunks | 7,531 fixed-size chunks (local artifacts) | Minor mismatch |
| Subsample best baseline MRR | Hybrid ≈ 0.654 | GraphRAG ≈ 0.580, Dense ≈ 0.540, Hybrid ≈ 0.462 | Partial mismatch |
| Full-corpus Dense MRR | 0.166 (reported best baseline) | 0.166 | Match (value) |
| Full-corpus baseline ranking | Dense reported strongest baseline | GraphRAG/ReRank/Hybrid above Dense in reproduced run | Mismatch |
| Full-corpus orchestration best | Confidence ≈ 0.205 (Step 3) | Confidence ≈ 0.209 | Close match |
| Full-corpus Voting MRR | ≈ 0.190 (Step 3), ≈ 0.189 (Step 2 section) | ≈ 0.202 | Near but higher |
| Full-corpus Waterfall MRR | ≈ 0.161 (Step 3) | ≈ 0.208 | Mismatch |
| Subsample → full trend | Metrics decrease on full corpus | Metrics decrease on full corpus | Match |
| Evaluation protocol | Shared IR metrics (P@k, Recall, MRR, plus nDCG in report) | Shared IR metrics (P@1/3/5/10, Recall@1/3/5/10, MRR) | Largely aligned |

**Interpretation:**  
The reproduction confirms the main scaling trend (performance drops from subsample to full corpus) and reproduces key anchor values such as full-corpus Dense MRR (~0.166). Differences in some absolute scores and method rankings are likely due to artifact/version variation, retriever serialization differences, and implementation-specific fusion/reranking settings.
